# Lab 5: Polygenic scores and probability distributions

## Learning objectives

By the end of this lab, you will be able to:

- distinguish discrete and continuous random variables and evaluate the assumptions of binomial, normal, and Poisson models;
- standardize a score using a stated reference cohort and compare empirical proportions with normal-model probabilities;
- reproduce substance-use prevalence by PGS quintile and examine how using deciles changes the displayed pattern;
- audit the participants included in a calculation and preserve unavailable scores and outcomes; and
- interpret synthetic results with appropriate limits on prediction, causation, and reuse.


### Connection to Exercise 5 and the Lab 5 warm-up

Exercise 5 introduced how polygenic scores are constructed and how a reference group gives meaning to a standardized score. The warm-up used Lab 5 results to examine regular-use prevalence across PGS quintiles. Here you will reproduce those results, check a normal approximation against observed data, and examine how grouping affects the patterns.

The main question is: **How can we describe a PGS distribution, and what do its group summaries show about regular alcohol and cannabis use?** A short binomial demonstration connects this work to Lab 4. A separate Poisson example applies probability models to daily event counts.

### Dataset provenance

The score in `instructional_pgs.csv` is a **synthetic instructional analog**. It is not an official ABCD score and was not built from real participant genotypes or real GWAS weights. The substance-use data are also synthetic. This notebook analyzes the supplied score; it does not construct a PGS from genetic variants.

All data analyses here are descriptive. They do not establish statistical significance, identify a genetic mechanism, or establish causation. Section 8 brings together the reporting and reuse limitations.


### Working in this notebook

Run each completed **Demo** cell and complete all four **Your Turn** activities. Replace the `None` placeholders with your own expressions. Later cells use those results, so complete each activity before continuing. Keep brief notes on the interpretation prompts; no separate written response in the notebook is required.

You may use GitHub Copilot as a tutor to explain code or concepts, interpret an error, or offer a hint. First compare an unfinished step with the preceding demonstration. Do not ask an AI tool to complete the notebook or fill in all the TODOs for you.

Complete the Canvas reflection independently using your notebook and permitted documentation. Do not use generative AI to answer, draft, rewrite, or check assessment responses, and do not paste assessment questions into an AI tool. Approved accessibility tools, spelling and grammar checkers, and instructor-authorized translation tools are permitted.


### The distribution roadmap

The variable's possible values suggest a model; its assumptions determine whether that model is useful.

| Random variable | Possible values | Type | Candidate model | Question |
| --- | --- | --- | --- | --- |
| Number of twins reporting recent cannabis use in one pair | 0, 1, 2 | Discrete count | Binomial | How does the observed distribution compare with an independent-trials model? |
| Externalizing PGS | Values on a quantitative score scale | Treated as continuous | Normal approximation | How closely do the observed center and upper tail follow a normal model? |
| Number of opioid-overdose hospitalizations in one day | 0, 1, 2, ... | Discrete count | Poisson | What probabilities follow from a specified event rate? |


## 1. Setup and required analysis

Use the same settings throughout the required lab: externalizing PGS, session `ses-07A`, and a regular-use threshold of at least 13 days in the past three months. Section 6 starts with the warm-up's quintiles, then compares them with deciles while holding the cohort and outcome definitions fixed.

Leave the settings below unchanged. Optional checks of other variables and upper-tail cutoffs appear in Section 10, after the required work.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from math import comb

# Paths are relative to this notebook in labs/. Keep source tables unchanged.
instructional_pgs = pd.read_csv("../data/labs_04_05/instructional_pgs.csv")
su_y_tlfb = pd.read_csv("../data/labs_04_05/su_y_tlfb.csv")
gn_y_genrel = pd.read_csv("../data/labs_04_05/gn_y_genrel.csv")

source_tables = {
    "instructional_pgs": instructional_pgs,
    "su_y_tlfb": su_y_tlfb,
    "gn_y_genrel": gn_y_genrel,
}
source_table_summary = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": table.shape[1]}
    for name, table in source_tables.items()
])
display(source_table_summary)


In [ ]:
# Required settings: leave these unchanged.
ANALYSIS_SESSION = "ses-07A"
PGS_VARIABLE = "pgs_externalizing_raw"
REGULAR_USE_DAYS = 13       # RESEARCHER DECISION, introduced in Lab 4
REFERENCE_DDOF = 0         # Spread of the defined reference cohort
NORMAL_TAIL_CUT = 2.0
PREVALENCE_AXIS_MAX = 0.15  # Common zero-based scale for both grouping displays

print("Required score       : synthetic externalizing PGS")
print("Substance-use session:", ANALYSIS_SESSION)
print("PGS outcome criterion: at least", REGULAR_USE_DAYS, "use days in 3 months")
print("Binomial outcome     : any recent cannabis use (more than 0 days)")
print("Normal-model check   : within 1 SD, within 2 SD, and z >", NORMAL_TAIL_CUT)
print("Grouping comparison  : quintiles, followed by deciles")


### Measures used in the two analyses

Lab 4 introduced both recent and regular use, then used recent use for its main twin-pair probability analysis. Keep that distinction here.

| Analysis | Outcome rule | Unit being summarized |
| --- | --- | --- |
| Binomial bridge, Section 2 | Recent cannabis use: use days > 0 | Complete twin pair |
| PGS groups, Section 6 | Regular alcohol or cannabis use: use days >= 13 | Participant with a PGS and an available substance outcome |

For both rules, an unavailable use-day value remains missing. The 13-day threshold is a researcher decision, not a diagnosis or evidence of use in every calendar week. A value below 13 can include some use.


### Prepare for the Canvas reflection

Evidence to review: Use the roadmap to identify each variable's possible values, type, and candidate model. Consider the assumptions as well as the values.


## 2. Binomial bridge from Lab 4

Lab 4 considered events such as "both twins used." Now let $X$ be the **number of twins in a pair reporting recent cannabis use**. We examine the entire distribution over $X=0,1,2$.

A binomial model requires a fixed number of trials, two outcomes per trial, the same outcome probability $p$ for each trial, and independent trials. With two twins, it predicts

$$P(X=k)={2 \choose k}p^k(1-p)^{2-k}.$$

The completed code rebuilds the unique-pair table and retains pairs with both outcomes available. For MZ and DZ pairs separately, it estimates a common $p$ from all twin members in those complete pairs. Here "success" means reporting recent cannabis use.

Lab 4 used separate A and B marginal proportions in $P(A)\times P(B)$. The binomial model instead uses a common $p$, so its expectation for $X=2$ is $p^2$. Those expectations can differ slightly when the A and B marginals differ.


In [ ]:
# Demo: run this completed cell. It rebuilds the Lab 4 pair table.

def build_pair_table(relatedness, keep_zygosity=(1, 2)):
    rows = relatedness.loc[
        relatedness["gn_y_genrel_id__paired__01"].notna()
        & relatedness["gn_y_genrel_zyg__01"].isin(keep_zygosity)
    ].copy()
    first = rows[["participant_id", "gn_y_genrel_id__paired__01"]].min(axis=1)
    second = rows[["participant_id", "gn_y_genrel_id__paired__01"]].max(axis=1)
    pairs = pd.DataFrame({
        "twin_a": first, "twin_b": second,
        "zygosity": rows["gn_y_genrel_zyg__01"].to_numpy(),
    }).drop_duplicates(subset=["twin_a", "twin_b"])
    pairs = pairs.sort_values(["twin_a", "twin_b"]).reset_index(drop=True)
    pairs["zygosity_label"] = pairs["zygosity"].map({1: "MZ", 2: "DZ"})
    return pairs
assert gn_y_genrel["participant_id"].is_unique, "Relatedness IDs must be unique."
pair_table = build_pair_table(gn_y_genrel)
pair_members = pd.concat([pair_table["twin_a"], pair_table["twin_b"]])
assert pair_members.is_unique, "A participant appears in more than one retained pair."
tlfb_wave = su_y_tlfb.loc[su_y_tlfb["session_id"] == ANALYSIS_SESSION].set_index("participant_id")
assert tlfb_wave.index.is_unique, "Select one substance-use row per participant."
cannabis_days = tlfb_wave["su_y_tlfb__mj__3mo_ud"]
# Three states preserved, exactly as in Lab 4.
recent_cannabis = (cannabis_days > 0).astype("Int64").where(cannabis_days.notna())
paired = pair_table.assign(
    a=recent_cannabis.reindex(pair_table["twin_a"]).to_numpy(),
    b=recent_cannabis.reindex(pair_table["twin_b"]).to_numpy(),
).dropna(subset=["a", "b"])
print("Complete pairs:", len(paired),
      "| MZ:", int((paired["zygosity_label"] == "MZ").sum()),
      "| DZ:", int((paired["zygosity_label"] == "DZ").sum()))


In [ ]:
# Demo: compare all three pair counts with a binomial model.
assumption_rows, probability_rows = [], []
for zygosity in ["MZ", "DZ"]:
    subset = paired.loc[paired["zygosity_label"] == zygosity]
    users_per_pair = subset["a"].astype(int) + subset["b"].astype(int)
    p_a = float(subset["a"].astype(float).mean())
    p_b = float(subset["b"].astype(float).mean())
    p = float(pd.concat([subset["a"], subset["b"]]).astype(float).mean())
    assumption_rows.append({
        "zygosity": zygosity, "complete pairs": len(subset),
        "twin members": 2 * len(subset),
        "A marginal": p_a, "B marginal": p_b, "common p": p,
    })
    for k in (0, 1, 2):
        probability_rows.append({
            "zygosity": zygosity, "users in pair": k,
            "observed": float((users_per_pair == k).mean()),
            "binomial expectation": comb(2, k) * p**k * (1 - p)**(2 - k),
        })
binomial_reference = pd.DataFrame(assumption_rows)
binomial_comparison = pd.DataFrame(probability_rows)
display(binomial_reference.round(4))
display(binomial_comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
width = 0.38
for ax, zygosity in zip(axes, ["MZ", "DZ"]):
    block = binomial_comparison.loc[binomial_comparison["zygosity"] == zygosity]
    positions = np.arange(3)
    ax.bar(positions - width / 2, block["observed"], width, label="Observed")
    ax.bar(positions + width / 2, block["binomial expectation"], width,
           label="Binomial expectation")
    ax.set_xticks(positions, ["0", "1", "2"])
    ax.set_xlabel("Twins reporting recent cannabis use")
    ax.set_title(f"{zygosity} pairs")
axes[0].set_ylabel("Proportion of complete pairs")
axes[0].legend()
plt.tight_layout()
plt.show()


### Interpret the comparison

For each zygosity group, which pair counts occur more often than the binomial model predicts, and which occur less often? Compare the size of the departures across the two panels.

Extra concordant pairs ($X=0$ or $X=2$) and fewer discordant pairs ($X=1$) describe co-twin outcome clustering. Independence is questionable for linked twins, and a common outcome probability can also be an oversimplification. These descriptive departures do not constitute a formal goodness-of-fit test or identify why twins resemble one another.


### Prepare for the Canvas reflection

Evidence to review: `binomial_reference` gives the complete-pair counts and common $p$. Use `binomial_comparison` and its plot to connect an observed departure to a model assumption.


## 3. PGS provenance, join, and denominator audit

The PGS file has one row per participant. The substance-use file has one row per participant per visit. Select `ses-07A` before joining so both inputs have participant-level rows keyed by `participant_id`.

| Provenance field | Value for this lab |
| --- | --- |
| Score | Synthetic externalizing PGS analog |
| Construction | Simulated score supplied for instruction; no participant genotypes or estimated GWAS weights are analyzed here |
| Target cohort | Synthetic ABCD-like cohort |
| Raw scale | Arbitrary scale representing a weighted-sum score |
| Standardization reference | All participants with an available PGS in the supplied score file, before joining or filtering outcomes |
| Official ABCD result | No |

A blank PGS represents an unavailable score, such as one unavailable after genetic quality control. It is not a score of zero and does not mean that a person has no genotype.

The completed workflow derives the regular-use indicators, validates a one-to-one join, and audits matched identifiers, available scores, and available outcomes separately.


In [ ]:
# Demo: run this completed cell.
substance_wave = su_y_tlfb.loc[su_y_tlfb["session_id"] == ANALYSIS_SESSION].copy()

assert substance_wave["participant_id"].is_unique, "session filter did not give one row per participant"
assert instructional_pgs["participant_id"].is_unique, "score table is not one row per participant"

# Three states, never a bare comparison, for both substances.
for column, derived in [("su_y_tlfb__alc__3mo_ud", "regular_alcohol"),
                        ("su_y_tlfb__mj__3mo_ud", "regular_cannabis")]:
    days = substance_wave[column]
    substance_wave[derived] = (days >= REGULAR_USE_DAYS).astype("Int64").where(days.notna())

display(substance_wave[["participant_id", "su_y_tlfb__alc__3mo_ud", "regular_alcohol",
                        "su_y_tlfb__mj__3mo_ud", "regular_cannabis"]].head(4))

In [ ]:
# Demo: run this completed join and denominator audit.
analysis_wave = instructional_pgs.merge(
    substance_wave, on="participant_id", how="inner", validate="one_to_one"
)

has_score = analysis_wave[PGS_VARIABLE].notna()

denominator_audit = pd.DataFrame({
    "stage": [
        "rows in the score table",
        "participants with a nonmissing PGS",
        "participants in the selected substance-use wave",
        "participant identifiers matched across the two tables",
        "matched participants with a nonmissing PGS",
        "matched, with PGS and alcohol available",
        "matched, with PGS and cannabis available",
    ],
    "n": [
        len(instructional_pgs),
        int(instructional_pgs[PGS_VARIABLE].notna().sum()),
        len(substance_wave),
        len(analysis_wave),
        int(has_score.sum()),
        int((has_score & analysis_wave["regular_alcohol"].notna()).sum()),
        int((has_score & analysis_wave["regular_cannabis"].notna()).sum()),
    ],
})

display(denominator_audit)

### Read the audit

Compare matched identifiers with matched participants who have a usable PGS. A successful join matches people; it does not fill in their missing scores. The final two rows give separate usable denominators for alcohol and cannabis. The two usable denominators may differ because alcohol and cannabis responses can be unavailable independently. Treat each unavailable outcome as missing, never as zero use days.


## 4. Standardize the score

Apply the relationship from Exercise 5:

$$z=\frac{x-\text{reference mean}}{\text{reference SD}}.$$

Use the **full synthetic scored cohort before the join or outcome filtering**. `REFERENCE_DDOF = 0` calculates the spread of this defined cohort; it sets the scale for the lab. Keep those reference values fixed when analyzing outcomes.

Exercise 5 used a rounded mean of **12.4** and SD of **3.1**. Here, calculate both from the source data and use their **unrounded values** in the arithmetic. Missing scores remain missing. A z-score describes direction and distance from this reference mean in SD units.


In [ ]:
# Demo: run this completed cell.
pgs_series = instructional_pgs.set_index("participant_id")[PGS_VARIABLE]

raw_summary = pd.DataFrame({
    "statistic": ["n scored", "n missing", "mean", "std dev", "min", "25%", "median", "75%", "max"],
    "value": [
        int(pgs_series.notna().sum()), int(pgs_series.isna().sum()),
        round(pgs_series.mean(), 3), round(pgs_series.std(ddof=REFERENCE_DDOF), 3),
        round(pgs_series.min(), 3), round(pgs_series.quantile(0.25), 3),
        round(pgs_series.median(), 3), round(pgs_series.quantile(0.75), 3),
        round(pgs_series.max(), 3),
    ],
})

display(raw_summary)

# Worked coding pattern on illustrative values, before using the full PGS.
demo_raw = pd.Series([8.0, 10.0, 12.0])
demo_mean = demo_raw.mean()
demo_sd = demo_raw.std(ddof=REFERENCE_DDOF)
demo_z = (demo_raw - demo_mean) / demo_sd
print("Coding pattern for standardization (illustrative values):")
display(pd.DataFrame({"raw": demo_raw, "z": demo_z}).round(4))


### Your Turn 1: standardize the PGS

Complete the three TODOs: calculate the reference mean, calculate the reference SD using `ddof=REFERENCE_DDOF`, and standardize `pgs_series`. Keep its participant index and missing values so the scores can be attached to the correct people later.


In [ ]:
# Your Turn 1: standardize the PGS.
# TODO: replace each None with the requested expression.

reference_values = pgs_series.dropna()   # the full scored cohort

# TODO: calculate the mean of reference_values
reference_mean = None
# TODO: calculate the standard deviation using ddof=REFERENCE_DDOF
reference_sd = None
# TODO: standardize the full pgs_series with those reference values
pgs_z = None

assert reference_mean is not None and reference_sd is not None and pgs_z is not None, (
    "Complete all three TODOs in Your Turn 1 before continuing."
)
assert isinstance(pgs_z, pd.Series), "pgs_z must be a pandas Series."
assert pgs_z.index.equals(pgs_series.index), "Keep the participant index."
assert pgs_z.isna().equals(pgs_series.isna()), "Preserve unavailable scores."
assert reference_sd > 0, "The reference SD must be positive."

print("Reference sample: full synthetic scored cohort")
print("Reference sample n :", len(reference_values))
print("Reference mean     :", round(reference_mean, 4))
print("Reference SD       :", round(reference_sd, 4))
print()
print("Standardized score in the reference sample:",
      f"mean {pgs_z.dropna().mean():+.4f},",
      f"sd {pgs_z.dropna().std(ddof=REFERENCE_DDOF):.4f}")


In [ ]:
# Demo: compare scales and select three illustrative participants.
raw_scored = pgs_series.dropna()
ranks_match = raw_scored.rank().equals(pgs_z.loc[raw_scored.index].rank())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].hist(raw_scored, bins=40)
axes[0].axvline(reference_mean, linewidth=2)
axes[0].set_title("Raw PGS")
axes[0].set_xlabel("Raw score units")
axes[0].set_ylabel("Participants")
axes[1].hist(pgs_z.dropna(), bins=40)
axes[1].axvline(0, linewidth=2)
axes[1].set_title("Standardized PGS")
axes[1].set_xlabel("Standard deviations from the reference mean")
plt.tight_layout()
plt.show()
print("Ranks preserved by standardization:", ranks_match)

example_rows = []
for label, target_z in [("Below the mean", -1), ("Near the mean", 0),
                         ("Above the mean", 1)]:
    example_id = (pgs_z - target_z).abs().idxmin()
    example_rows.append({
        "example": label, "raw score": pgs_series.loc[example_id],
        "z-score": pgs_z.loc[example_id],
    })
standardization_examples = pd.DataFrame(example_rows)
display(standardization_examples.round(4))


### Interpret the scales

Compare the axes and shapes. Standardization changes the units, center, and spread while preserving shape and rank. Interpret one row of `standardization_examples`, naming the full scored cohort as its reference. The next section asks whether a normal model is a useful approximation.


### Prepare for the Canvas reflection

Evidence to review: Use `denominator_audit`, `raw_summary`, the printed reference statistics, and `standardization_examples` to distinguish a matched row, an available score, and a standardized position.


## 5. Does a normal model describe the PGS?

Exercise 5 asked you to assume that a normal model fit the distribution. Now compare that model with the observed PGS values. A standardized mean near zero and SD near one follow from the calculation; they do not establish normality.

Use `pgs_z` to calculate the observed proportions within one SD, within two SDs, and above $z=2$. The completed code supplies the corresponding standard-normal probabilities. These are proportions of scores in regions of a distribution, not probabilities of substance use.


In [ ]:
# Demo: identify the values used in the required normal-model check.
pgs_normal_z = pgs_z.dropna()
normal_reference = pd.DataFrame([{
    "variable": "Synthetic externalizing PGS",
    "reference": "Full scored cohort, before outcome filtering",
    "n": len(pgs_normal_z),
    "distinct raw scores": pgs_series.nunique(),
    "upper-tail rule": "z > 2",
}])
display(normal_reference)

# Worked coding pattern on five illustrative z-scores, not the PGS results.
demo_z_values = pd.Series([-2.2, -0.5, 0.4, 1.5, 2.3])
print("Coding pattern for empirical proportions (illustrative values):")
print("Within 1 SD:", (demo_z_values.abs() <= 1).mean())
print("Within 2 SD:", (demo_z_values.abs() <= 2).mean())
print("Above z = 2:", (demo_z_values > NORMAL_TAIL_CUT).mean())


### Your Turn 2: compare empirical proportions with a normal model

Complete the three TODOs using `pgs_normal_z`. Count observations on the boundary for the central intervals (`<=`), but use a strict upper-tail rule (`> 2`). A Boolean condition followed by `.mean()` gives its observed proportion when every value is available.


In [ ]:
# Your Turn 2: empirical proportions against normal benchmarks.
# TODO: replace each None with the requested expression.

# TODO: proportion of pgs_normal_z with absolute value <= 1
within_one_sd = None
# TODO: proportion of pgs_normal_z with absolute value <= 2
within_two_sd = None
# TODO: proportion of pgs_normal_z strictly greater than NORMAL_TAIL_CUT
observed_tail = None

assert all(value is not None for value in [within_one_sd, within_two_sd, observed_tail]), (
    "Complete all three TODOs in Your Turn 2 before continuing."
)

normal_within_one = stats.norm.cdf(1) - stats.norm.cdf(-1)
normal_within_two = stats.norm.cdf(2) - stats.norm.cdf(-2)
normal_tail = stats.norm.sf(NORMAL_TAIL_CUT)

normal_comparison = pd.DataFrame({
    "region": ["-1 <= z <= 1", "-2 <= z <= 2", "z > 2"],
    "normal model": [normal_within_one, normal_within_two, normal_tail],
    "observed": [within_one_sd, within_two_sd, observed_tail],
})
normal_comparison["observed minus model"] = (
    normal_comparison["observed"] - normal_comparison["normal model"]
)
display(normal_comparison.round(4))


In [ ]:
# Demo: compare the empirical density with a standard normal curve.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(pgs_normal_z, bins=40, density=True, alpha=0.55, label="Observed PGS")
grid = np.linspace(min(-4, pgs_normal_z.min()), max(4, pgs_normal_z.max()), 400)
ax.plot(grid, stats.norm.pdf(grid), linewidth=2, label="Standard normal model")
for edge in (-2, -1, 1):
    ax.axvline(edge, linestyle=":", linewidth=1)
ax.axvline(NORMAL_TAIL_CUT, linewidth=2, color="firebrick", label="Upper-tail cutoff: z = 2")
ax.set_xlabel("Standardized externalizing PGS")
ax.set_ylabel("Density")
ax.set_title("Externalizing PGS and a normal approximation")
ax.legend()
plt.tight_layout()
plt.show()


### Interpret the model comparison

Use the histogram and numerical comparisons to judge how well the normal model describes the center and upper tail of this score distribution. Support your judgment with the observed and model proportions. Agreement in one region alone does not establish a good approximation everywhere.

The histogram's vertical axis is **density**; area over an interval represents a proportion. This comparison assesses an approximation descriptively, rather than proving that the scores follow a normal distribution. Rank-based groups in Section 6 do not require normality.


### Prepare for the Canvas reflection

Evidence to review: `normal_comparison` and the histogram distinguish observed proportions from model probabilities. Section 10 offers optional checks of other variables and upper-tail cutoffs.


## 6. Reproduce the warm-up and compare PGS groupings

Start with the warm-up's five PGS groups. The completed code assigns quintiles by rank in the **full scored cohort**, then attaches those labels to the selected substance-use wave. Standardizing preserves the ranks, so raw and standardized PGS values produce the same groups.

For each substance, the prevalence denominator includes only participants in that group with the outcome available. The numerator counts those meeting the regular-use criterion within that same denominator. After reproducing the quintile results, you will compare them with ten groups, or deciles.


In [ ]:
# Demo: assign quintiles before joining or filtering outcomes.
scored_cohort = pgs_z.dropna()
quintile_groups = pd.qcut(scored_cohort, q=5, labels=range(1, 6))

analysis_frame = analysis_wave.set_index("participant_id").copy()
analysis_frame["pgs_z"] = pgs_z.reindex(analysis_frame.index)
analysis_frame["pgs_quintile"] = quintile_groups.reindex(analysis_frame.index)
analysis_frame = analysis_frame.loc[analysis_frame["pgs_z"].notna()].copy()

quintile_membership = (
    quintile_groups.value_counts().sort_index()
    .rename_axis("PGS quintile").reset_index(name="scored-cohort participants")
)
print("Full scored cohort:", len(scored_cohort))
print("Scored participants with a selected-wave record:", len(analysis_frame))
print("The two populations are identical here:",
      set(scored_cohort.index) == set(analysis_frame.index))
display(quintile_membership)

raw_quintiles = pd.qcut(pgs_series.dropna(), q=5, labels=range(1, 6))
print("Raw and standardized PGS give identical quintiles:",
      raw_quintiles.equals(quintile_groups))


# Worked coding pattern: remove unavailable outcomes, then count and divide.
demo_outcomes = pd.Series([1, 0, 0, pd.NA], dtype="Int64")
demo_available = demo_outcomes.dropna()
demo_denominator = len(demo_available)
demo_meeting_criterion = int((demo_available == 1).sum())
demo_prevalence = demo_meeting_criterion / demo_denominator
print("Coding pattern for group prevalence (illustrative outcomes):")
display(pd.DataFrame([{
    "denominator": demo_denominator,
    "meeting criterion": demo_meeting_criterion,
    "prevalence": demo_prevalence,
}]).round(4))


Each quintile contains approximately 20% of the scored cohort. That percentage describes group membership, not regular-use prevalence. These are empirical rank groups; tied scores can make group sizes unequal.


### Your Turn 3: reproduce substance-specific prevalence by quintile

For each substance and quintile, complete the three TODOs: the available-outcome denominator, the number meeting the regular-use criterion, and their prevalence. `values` contains the available 0/1 outcomes for that substance in that quintile.


In [ ]:
# Your Turn 3: substance-specific prevalence within PGS quintiles.
# TODO: replace each None with the requested expression.

quintile_rows = []
for outcome, label in [("regular_alcohol", "alcohol"), ("regular_cannabis", "cannabis")]:
    for group_value in range(1, 6):
        values = analysis_frame.loc[
            analysis_frame["pgs_quintile"] == group_value, outcome
        ].dropna()
        quintile_rows.append({
            "substance": label,
            "group": group_value,
            # TODO: number with this outcome available in this quintile
            "denominator": None,
            # TODO: number meeting the regular-use threshold
            "meeting criterion": None,
            # TODO: number meeting the criterion divided by the denominator
            "prevalence": None,
        })

quintile_prevalence = pd.DataFrame(quintile_rows)
assert quintile_prevalence[["denominator", "meeting criterion", "prevalence"]].notna().all().all(), (
    "Complete all three TODOs in Your Turn 3 before continuing."
)
quintile_prevalence["prevalence"] = quintile_prevalence["prevalence"].astype(float)
display(quintile_prevalence.round({"prevalence": 4}))


In [ ]:
# Demo: reproduce both warm-up patterns with the same zero-based y-axis.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, label in zip(axes, ["alcohol", "cannabis"]):
    block = quintile_prevalence.loc[quintile_prevalence["substance"] == label]
    ax.plot(block["group"], block["prevalence"], marker="o")
    for _, row in block.iterrows():
        ax.annotate(f"n={int(row['denominator'])}", (row["group"], row["prevalence"]),
                    textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
    ax.set_ylim(0, PREVALENCE_AXIS_MAX)
    ax.set_xticks(range(1, 6))
    ax.set_xlabel("PGS quintile (lowest to highest)")
    ax.set_title(f"Regular {label} use")
axes[0].set_ylabel("Observed prevalence")
fig.suptitle("Regular use by externalizing PGS quintile")
plt.tight_layout()
plt.show()


### Read the reproduced results

Compare `quintile_prevalence` and both panels with the warm-up. Identify the numerator and denominator behind one point. Then use the following counts to explain why the score groups do not perfectly separate people who meet the regular-use criterion from those who do not.


In [ ]:
# Demo: retain one compact summary of outcome overlap at the extremes.
overlap_rows = []
for outcome, label in [("regular_alcohol", "alcohol"), ("regular_cannabis", "cannabis")]:
    low = analysis_frame.loc[analysis_frame["pgs_quintile"] == 1, outcome].dropna()
    high = analysis_frame.loc[analysis_frame["pgs_quintile"] == 5, outcome].dropna()
    overlap_rows.append({
        "substance": label,
        "Q1 meeting criterion": int((low == 1).sum()),
        "Q5 not meeting criterion": int((high == 0).sum()),
        "Q5 available outcomes": len(high),
    })
overlap_summary = pd.DataFrame(overlap_rows)
display(overlap_summary)


### Demo: compare quintiles with deciles

The next completed cell changes only the number of PGS groups. It uses the same reference cohort, visit, regular-use threshold, and substance-specific availability rules. Both sets of results remain available as `quintile_prevalence` and `decile_prevalence`.

Deciles provide finer divisions of score rank and smaller denominators. Their proportions may fluctuate more from group to group. Keep the shared plotting scale and report both substances rather than choosing whichever grouping produces a smoother pattern.


In [ ]:
# Demo: form deciles independently, leaving the quintile results unchanged.
decile_groups = pd.qcut(scored_cohort, q=10, labels=range(1, 11))
decile_frame = analysis_frame.copy()
decile_frame["pgs_decile"] = decile_groups.reindex(decile_frame.index)

decile_rows = []
for outcome, label in [("regular_alcohol", "alcohol"), ("regular_cannabis", "cannabis")]:
    for group_value in range(1, 11):
        available = decile_frame.loc[
            decile_frame["pgs_decile"] == group_value, outcome
        ].dropna()
        denominator = len(available)
        meeting_criterion = int((available == 1).sum())
        decile_rows.append({
            "substance": label, "group": group_value,
            "denominator": denominator, "meeting criterion": meeting_criterion,
            "prevalence": meeting_criterion / denominator if denominator else np.nan,
        })
decile_prevalence = pd.DataFrame(decile_rows)
display(decile_prevalence.round({"prevalence": 4}))

comparison_rows = []
for grouping, summary in [("Quintiles", quintile_prevalence), ("Deciles", decile_prevalence)]:
    for substance in ["alcohol", "cannabis"]:
        block = summary.loc[summary["substance"] == substance].sort_values("group")
        comparison_rows.append({
            "substance": substance, "grouping": grouping,
            "smallest denominator": int(block["denominator"].min()),
            "largest denominator": int(block["denominator"].max()),
            "lowest-group prevalence": block["prevalence"].iloc[0],
            "highest-group prevalence": block["prevalence"].iloc[-1],
            "downward steps": int((block["prevalence"].diff().dropna() < 0).sum()),
        })
grouping_comparison = pd.DataFrame(comparison_rows)
display(grouping_comparison.round(4))


In [ ]:
# Demo: common y-axis across all panels; each row spans the full score ranking.
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
for row_index, (grouping, summary, count) in enumerate([
    ("quintile", quintile_prevalence, 5), ("decile", decile_prevalence, 10)
]):
    for column_index, substance in enumerate(["alcohol", "cannabis"]):
        ax = axes[row_index, column_index]
        block = summary.loc[summary["substance"] == substance]
        ax.plot(block["group"], block["prevalence"], marker="o")
        ax.set_ylim(0, PREVALENCE_AXIS_MAX)
        ax.set_xticks(range(1, count + 1))
        ax.set_xlabel(f"PGS {grouping} (lowest to highest)")
        ax.set_title(f"Regular {substance} use: {count} groups")
        if column_index == 0:
            ax.set_ylabel("Observed prevalence")
fig.suptitle("Same cohort and outcome definitions, different PGS groupings")
plt.tight_layout()
plt.show()


### Interpret the grouping comparison

For each substance, what happens to the denominators and the number of downward steps when you use deciles? Does the overall direction change? Use the table and all four panels; a comparison of the endpoints alone can miss reversals between them.

The lowest and highest quintiles each cover about 20% of the score ranking, whereas the extreme deciles each cover about 10%. Their endpoint prevalences therefore summarize different sets of participants. A more irregular decile pattern need not contradict an overall upward quintile pattern.


### Prepare for the Canvas reflection

Evidence to review: Use `quintile_prevalence` for the warm-up replication, `grouping_comparison` and the four-panel figure for the grouping comparison, and `overlap_summary` for limits on individual prediction. A PGS z-score or rank group describes score position; observed prevalence describes outcomes within an eligible group. Neither supplies a validated outcome probability for a new individual or a causal effect.


## 7. Poisson as a transfer application

The binomial model counted outcomes in a fixed pair. Here we count events in a fixed **time interval**. This fictional example is separate from the PGS analysis.

Imagine a county averaging $\lambda=2.4$ opioid-overdose hospitalizations per day. Let $X$ count hospitalization **events in one day**, rather than unique people:

$$X\sim\operatorname{Poisson}(\lambda=2.4).$$

A basic Poisson model assumes a stable rate and independent event occurrences over the interval. Its expected count and variance both equal $\lambda$. At a stable rate, the expected count over seven days is $7\lambda$.

In SciPy, `stats.poisson.pmf(k, LAMBDA)` gives $P(X=k)$ and `stats.poisson.cdf(k, LAMBDA)` gives $P(X\leq k)$. For $P(X\geq5)$, use the complement of $P(X\leq4)$.


In [ ]:
# Demo: run this completed cell.
LAMBDA = 2.4
INTERVAL = "one day"

theoretical = pd.DataFrame({
    "count": range(0, 9),
    "P(X = k)": [round(stats.poisson.pmf(k, LAMBDA), 4) for k in range(0, 9)],
})

print(f"X = number of opioid-overdose hospitalizations in {INTERVAL}")
print(f"lambda = {LAMBDA} events per day (fictional)")
display(theoretical)

# Worked coding pattern with different boundaries from Your Turn 4.
print("Coding pattern for Poisson cumulative probabilities:")
print("P(X <= 1) in one day:", round(stats.poisson.cdf(1, LAMBDA), 4))
print("P(X >= 2) in one day:", round(1 - stats.poisson.cdf(1, LAMBDA), 4))
print("Expected events in 2 days:", 2 * LAMBDA)


In [ ]:
# Demo: run this completed cell. The seed makes the simulation reproducible.
rng = np.random.default_rng(505)
simulated_days = rng.poisson(LAMBDA, size=365)

print("Simulated 365 days")
print("  mean     :", round(simulated_days.mean(), 3))
print("  variance :", round(simulated_days.var(ddof=0), 3))
print("  lambda   :", LAMBDA, " <- a basic Poisson model expects mean and variance to match")

counts = np.arange(0, simulated_days.max() + 1)
observed_share = [float((simulated_days == k).mean()) for k in counts]

plt.bar(counts, observed_share, alpha=0.6, label="simulated 365 days")
plt.plot(counts, stats.poisson.pmf(counts, LAMBDA), "o-", label="theoretical Poisson")
plt.xlabel("Hospitalizations in one day")
plt.ylabel("Proportion of days")
plt.title(f"Simulated daily counts against Poisson(lambda = {LAMBDA})")
plt.legend()
plt.show()

### Your Turn 4: calculate Poisson quantities

Complete the four TODOs using the helper or SciPy: $P(X=0)$, $P(X\leq2)$, $P(X\geq5)$ using the complement rule, and the expected seven-day count. The first three are probabilities for one day; the last is an expected number of events.


In [ ]:
# Your Turn 4: calculate four Poisson quantities.
# TODO: replace each None with the requested expression.

def poisson_probability(k, rate=LAMBDA):
    """P(X = k) for a Poisson variable with the given rate."""
    return stats.poisson.pmf(k, rate)


# TODO: probability of exactly zero hospitalizations in one day
p_zero = None
# TODO: probability of two or fewer in one day
p_two_or_fewer = None
# TODO: probability of five or more in one day, using the complement rule
p_five_or_more = None
# TODO: expected number across seven days
expected_week = None

assert all(value is not None for value in [p_zero, p_two_or_fewer, p_five_or_more, expected_week]), (
    "Complete all four TODOs in Your Turn 4 before continuing."
)

poisson_answers = pd.DataFrame({
    "quantity": ["P(X = 0)", "P(X <= 2)", "P(X >= 5)", "expected count in 7 days"],
    "value": [round(p_zero, 4), round(p_two_or_fewer, 4),
              round(p_five_or_more, 4), round(expected_week, 2)],
})

display(poisson_answers)


### Interpret the model and simulation

Interpret one calculated probability, naming both the event count and the one-day interval. Explain why the expected weekly count can be noninteger even though each observed count is a whole number.

The completed simulation was **generated from a Poisson model**. Its sample mean and variance should be reasonably close to $\lambda$, but need not match exactly. That agreement illustrates the model; it does not validate it for real hospitalization data. Changing daily rates or clustering in time can undermine its assumptions. Repeated hospitalizations of the same person can also introduce dependence among events.


### Prepare for the Canvas reflection

Evidence to review: Use `poisson_answers`, the stated daily rate, and the simulation display. Distinguish probabilities from expected counts, and explain how a plausible real-world condition could challenge the model's assumptions.


## 8. Provenance and responsible interpretation

**Reporting.** These synthetic results cannot be reported as ABCD findings or evidence about real participants. Describe the outcome definition, reference cohort, eligible denominators, and observed overlap. A descriptive group pattern does not determine an individual's outcome or identify a causal effect.

**Portability.** For a real PGS, documentation must identify the phenotype, discovery population and weights, target population, processing, and standardization reference. Differences in genetic variation, measurement, environments, or sampling can change performance across populations. Ancestry is not interchangeable with race, and these analyses do not support racial or deterministic claims.

**Authorized reuse.** Real genetic data can remain identifying after obvious identifiers are removed. Reuse must follow consent, approved purposes, data-use agreements, and access and reporting requirements. Protect files and outputs; being able to open a file does not establish permission to use it for a new question.

### Prepare for the Canvas reflection

Evidence to review: Return to the Section 3 provenance table. Identify a documented feature that limits how these results may be described or reused.


## 9. Finish the required lab

Before completing the Canvas reflection:

1. Complete all **four Your Turn** coding activities, then restart the kernel and run the required sections from top to bottom.
2. Confirm that unavailable scores and outcomes remain missing and that the source DataFrames remain unchanged.
3. Check the fixed reference cohort and the separate substance denominators. Keep both quintile and decile outputs available.
4. Review the named outputs below and your brief interpretation notes. Calculate with unrounded values; tables display probabilities and prevalences to four decimal places. Follow the assessment's entry instructions when reporting answers.
5. Save the notebook for your own use. The notebook itself is not submitted.

Optional Section 10 is ungraded and is not needed for the assessment. Its code is disabled by default, so a final Run All can safely include it.


In [ ]:
# Demo: confirm the settings and main output tables for your completed lab.
print("REQUIRED ANALYSIS")
print("  score               :", PGS_VARIABLE)
print("  session             :", ANALYSIS_SESSION)
print("  regular-use rule    : at least", REGULAR_USE_DAYS, "days in 3 months")
print("  reference cohort n  :", len(reference_values))
print("  fixed normal tail   : z >", NORMAL_TAIL_CUT)
print("  grouping outputs    : quintiles AND deciles")

output_index = pd.DataFrame({
    "section": [2, 3, 4, 5, 6, 6, 6, 7],
    "output": ["binomial_reference; binomial_comparison", "denominator_audit",
               "raw_summary; standardization_examples", "normal_reference; normal_comparison",
               "quintile_prevalence", "decile_prevalence; grouping_comparison",
               "overlap_summary", "poisson_answers"],
    "purpose": ["Pair denominators and model departures", "Matching and availability",
                "Reference statistics and score positions", "Observed versus normal-model proportions",
                "Warm-up replication", "Effect of changing the grouping",
                "Outcome overlap at the extremes", "Daily probabilities and weekly expectation"],
})
display(output_index)


## 10. Optional transfer checks

The required work ends above. These completed demonstrations let you investigate other distributions or upper-tail boundaries without changing `pgs_z`, the required normal comparison, or either prevalence table.

To explore, change `RUN_OPTIONAL_TRANSFER` to `True` and select one variable below. The PGS uses its original full scored-cohort reference. Each other variable uses all available values at `ses-07A`, with the same `ddof=0` convention. The `optional_` outputs belong only to this investigation.

Compare the histogram and the proportions within one and two SDs. Then inspect the upper-tail results at $z>1$, $z>1.5$, and $z>2$. Does your assessment depend on which region you examine? For a behavioral sum score, consider its bounds and repeated values when judging a continuous normal approximation.


In [ ]:
# Optional only: leave False for the normal required Run All.
RUN_OPTIONAL_TRANSFER = False
OPTIONAL_VARIABLE = "mh_y_upps__nurg_sum"

OPTIONAL_MENU = {
    "pgs_externalizing_raw": ("instructional_pgs", "Externalizing PGS"),
    "nc_y_nihtb__comp__tot__agecor_score": ("nc_y_nihtb", "Cognition composite, age-corrected"),
    "mh_y_upps__nurg_sum": ("mh_y_upps", "UPPS-P negative urgency sum"),
    "mh_p_cbcl__synd__ext_sum": ("mh_p_cbcl", "CBCL externalizing sum"),
}


In [ ]:
if RUN_OPTIONAL_TRANSFER:
    assert OPTIONAL_VARIABLE in OPTIONAL_MENU, "Choose an OPTIONAL_MENU variable."
    optional_table_name, optional_label = OPTIONAL_MENU[OPTIONAL_VARIABLE]
    if optional_table_name == "instructional_pgs":
        optional_values = pgs_series.dropna()
        optional_z = pgs_z.loc[optional_values.index].copy()
        optional_reference = "Full scored PGS cohort from Section 4"
    else:
        optional_source = pd.read_csv(f"../data/labs_04_05/{optional_table_name}.csv")
        optional_wave = optional_source.loc[optional_source["session_id"] == ANALYSIS_SESSION]
        assert optional_wave["participant_id"].is_unique
        optional_values = optional_wave.set_index("participant_id")[OPTIONAL_VARIABLE].dropna().astype(float)
        optional_sd = optional_values.std(ddof=REFERENCE_DDOF)
        assert optional_sd > 0, "A positive reference SD is needed."
        optional_z = (optional_values - optional_values.mean()) / optional_sd
        optional_reference = f"All available values at {ANALYSIS_SESSION}"

    print(optional_label)
    print("Reference:", optional_reference)
    print("Available values:", len(optional_values), "| Distinct values:", optional_values.nunique())
    optional_normal_comparison = pd.DataFrame({
        "region": ["-1 <= z <= 1", "-2 <= z <= 2", "z > 2"],
        "observed": [(optional_z.abs() <= 1).mean(), (optional_z.abs() <= 2).mean(),
                     (optional_z > 2).mean()],
        "normal model": [stats.norm.cdf(1) - stats.norm.cdf(-1),
                         stats.norm.cdf(2) - stats.norm.cdf(-2), stats.norm.sf(2)],
    })
    display(optional_normal_comparison.round(4))
    optional_tail_comparison = pd.DataFrame({
        "upper cutoff": [1.0, 1.5, 2.0],
        "observed proportion": [float((optional_z > cut).mean()) for cut in (1.0, 1.5, 2.0)],
        "normal proportion": [stats.norm.sf(cut) for cut in (1.0, 1.5, 2.0)],
    })
    display(optional_tail_comparison.round(4))

    optional_fig, optional_ax = plt.subplots(figsize=(8, 4.5))
    optional_ax.hist(optional_z, bins=40, density=True, alpha=0.55, label="Observed values")
    optional_grid = np.linspace(min(-4, optional_z.min()), max(4, optional_z.max()), 400)
    optional_ax.plot(optional_grid, stats.norm.pdf(optional_grid), label="Standard normal model")
    for optional_cut in (1.0, 1.5, 2.0):
        optional_ax.axvline(optional_cut, linestyle=":", linewidth=1)
    optional_ax.set_xlabel("Standard deviations from the optional reference mean")
    optional_ax.set_ylabel("Density")
    optional_ax.set_title(f"Optional check: {optional_label}")
    optional_ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Optional transfer checks skipped. The required outputs above are unchanged.")
